# 04 - Corrective RAG with LangChain v1 Middleware

## 🎯 Learning Objectives

By the end of this notebook, you will:
- Understand Corrective RAG (CRAG) architecture with self-correction
- Master LangChain v1's middleware system (`wrap_tool_call`, `before_model`, `wrap_model_call`)
- Implement document grading as tool call middleware
- Add web search fallback when retrieved documents are irrelevant
- Compare two middleware styles: Wrap-Style vs Node-Style

## ⚖️ What is Corrective RAG?

**Corrective RAG (CRAG)** adds a **self-correction layer** to RAG by validating retrieval quality and falling back to alternative sources when needed:

```mermaid
graph LR;
    A[Question] --> B[Agent];
    B --> C[RAG Tool];
    C --> D[Grade Documents];
    D -->|Relevant| E[Generate Answer];
    D -->|Irrelevant| F[Web Search Fallback];
    F --> E;
```

> **Note**: CRAG is a form of **Agentic RAG**. The diagram above highlights the corrective flow. In practice, the agent may invoke tools repeatedly in a loop (e.g., multiple RAG queries with different search terms) before reaching a final answer.

### LangChain v1 Middleware Hooks Used

| Hook | Purpose | Style |
|------|---------|-------|
| `wrap_tool_call` | Grade RAG tool outputs | Both |
| `before_model` | Inject fallback results / Apply state updates | Both |
| `wrap_model_call` | Filter tools dynamically | Wrap-Style only |

### Key Benefits:

- **Cleaner Architecture**: Middleware intercepts at precise points
- **Composable**: Stack multiple middleware for different concerns
- **Maintainable**: Each middleware has single responsibility
- **Testable**: Middleware can be tested in isolation

## 🔧 Setup

In [1]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
print("✅ Environment loaded")

✅ Environment loaded


In [2]:
from langchain_dev_utils.chat_models import register_model_provider, load_chat_model
from langchain_dev_utils.embeddings import register_embeddings_provider, load_embeddings

SILICONFLOW_BASE_URL = os.getenv("SILICONFLOW_BASE_URL", "https://api.siliconflow.cn/v1")

register_model_provider(
    provider_name="siliconflow",
    chat_model="openai-compatible",
    base_url=SILICONFLOW_BASE_URL,
)

register_embeddings_provider(
    provider_name="siliconflow",
    embeddings_model="openai-compatible",
    base_url=SILICONFLOW_BASE_URL,
)

CHAT_MODEL_NAME = os.getenv("SILICONFLOW_CHAT_MODEL", "Qwen/Qwen3-8B")
EMBEDDING_MODEL_NAME = os.getenv("SILICONFLOW_EMBEDDING_MODEL", "BAAI/bge-m3")

chat_model = load_chat_model(f"siliconflow:{CHAT_MODEL_NAME}")
embeddings = load_embeddings(f"siliconflow:{EMBEDDING_MODEL_NAME}")

print(f"✅ Models loaded: {CHAT_MODEL_NAME}")

✅ Models loaded: zai-org/GLM-4.5-Air


In [3]:
from langchain_oceanbase.vectorstores import OceanbaseVectorStore

connection_args = {
    "host": os.getenv("OCEANBASE_HOST", "127.0.0.1"),
    "port": int(os.getenv("OCEANBASE_PORT", "2881")),
    "user": os.getenv("OCEANBASE_USER", "root@test"),
    "password": os.getenv("OCEANBASE_PASSWORD", ""),
    "db_name": os.getenv("OCEANBASE_DB", "test"),
}

vector_store = OceanbaseVectorStore(
    embedding_function=embeddings,
    table_name="langchain_knowledge_base",
    connection_args=connection_args,
    vidx_metric_type="cosine",
    drop_old=False,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print("✅ Vector store and retriever ready")

✅ Vector store and retriever ready


## 🛠️ Step 1: Create RAG Retriever Tool

Build a retriever tool that searches Nike's 10-K report.

In [4]:
from langchain.tools import tool

@tool
def search_nike_report(query: str) -> str:
    """Search Nike's 10-K annual report for relevant business information.
    
    Use this tool when you need to find information about Nike's business,
    financials, operations, risks, or strategy from their official 10-K filing.
    """
    # Perform retrieval
    results = retriever.invoke(query)
    
    # Format results
    formatted_results = []
    for i, doc in enumerate(results, 1):
        page = doc.metadata.get('page', 'Unknown')
        formatted_results.append(
            f"Document {i} [Page {page}]:\n{doc.page_content}\n"
        )
    
    return "\n".join(formatted_results)

print("✅ RAG retriever tool created")
print(f"\n📚 Tool: {search_nike_report.name}")
print(f"📝 Description: {search_nike_report.description}")

✅ RAG retriever tool created

📚 Tool: search_nike_report
📝 Description: Search Nike's 10-K annual report for relevant business information.

Use this tool when you need to find information about Nike's business,
financials, operations, risks, or strategy from their official 10-K filing.


## 📊 Step 2: Document Grading with Structured Output

Create a grader that evaluates document relevance using structured outputs.

In [5]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# Define grading schema
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""
    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )
    reasoning: str = Field(
        description="Brief explanation for the relevance decision"
    )

# Grading prompt
grade_prompt = ChatPromptTemplate.from_template(
    """You are a grader assessing relevance of retrieved documents to a user question.

Retrieved document:

{document}

User question:

{question}

If the document contains keywords or semantic meaning related to the question, grade it as relevant.
Give a binary score 'yes' or 'no' to indicate whether the document is relevant to the question.
Also provide a brief reasoning for your decision."""
)

# Create grader with structured output
grader = grade_prompt | chat_model.with_structured_output(GradeDocuments)

print("✅ Document grader created")

✅ Document grader created


In [6]:
# Test document grader
print("🧪 Testing Document Grader")
question = "What are Nike's revenue sources?"

result_1 = grader.invoke({"question": question, "document": "Nike generates revenue from footwear, apparel, and equipment sales."})
result_2 = grader.invoke({"question": question, "document": "The weather today is sunny with a chance of rain."})

print(f"   Relevant doc → {result_1.binary_score} ({result_1.reasoning[:50]}...)")
print(f"   Irrelevant doc → {result_2.binary_score} ({result_2.reasoning[:50]}...)")

🧪 Testing Document Grader
   Relevant doc → yes (The document directly addresses the user question ...)
   Irrelevant doc → no (The document contains information about weather co...)


## 🌐 Step 3: Tavily Web Search Tool

Set up Tavily as the fallback search tool when RAG documents are not relevant.

In [7]:
from langchain_community.tools import TavilySearchResults

# Initialize Tavily search tool
tavily_search = TavilySearchResults(
    max_results=3,
    # include_raw_content=True,  # Optional: include full page content
)

print("✅ Tavily web search tool created")
print(f"\n🔍 Tool: {tavily_search.name}")
print(f"📝 Description: {tavily_search.description}")

✅ Tavily web search tool created

🔍 Tool: tavily_search_results_json
📝 Description: A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. Input should be a search query.


/var/folders/f3/zw7t8v4s6j3gd6hyktrdg85r0000gn/T/ipykernel_94632/1075327655.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(


In [8]:
# Test Tavily search
print("🧪 Testing Tavily Search")
test_result = tavily_search.invoke({"query": "Nike latest news 2024"})
print(f"   Results: {len(test_result)} items")
print(f"   First title: {test_result[0]['title'][:60]}...")

🧪 Testing Tavily Search
   Results: 3 items
   First title: Nike CEO Elliott Hill rolls out latest senior leadership ove...


## 🔄 Step 4: Custom State for Corrective RAG

Define custom agent state to track grading results and control flow.

### State vs Context in LangChain v1

| Type | Purpose | Mutability | Definition |
|------|---------|------------|------------|
| **State** (`AgentState`) | Track mutable data during a run | Mutable | TypedDict extending `AgentState` |
| **Context** (`context_schema`) | Static dependencies like user ID | Immutable | `@dataclass` |

For Corrective RAG, we need **mutable state** to track:
- Whether documents were relevant
- Query for potential fallback
- Grading history

In [9]:
from typing import Optional, List
from typing_extensions import NotRequired
from langchain.agents import AgentState

class CorrectiveRAGState(AgentState):
    """Extended state for tracking corrective RAG flow.
    
    Inherits 'messages' from AgentState. Adds fields for grading and fallback tracking.
    """
    last_rag_query: NotRequired[Optional[str]]       # Query for web search fallback
    documents_relevant: NotRequired[Optional[bool]]  # Grading result
    grade_reasoning: NotRequired[Optional[str]]      # Grading explanation
    used_web_fallback: NotRequired[bool]             # Whether web search was triggered
    grading_history: NotRequired[List[dict]]         # Debug trail

print("✅ CorrectiveRAGState defined (extends AgentState with grading/fallback fields)")

✅ CorrectiveRAGState defined (extends AgentState with grading/fallback fields)


## 🎯 Step 5: Document Grading Middleware (Multiple Hooks)

This middleware demonstrates that **one middleware class can implement multiple hooks**:

| Hook | Purpose |
|------|---------|
| `before_model` | Apply pending grading results to state, log current state |
| `after_model` | Log model response (tool calls or final answer) |
| `wrap_tool_call` | Execute RAG tool, grade documents, store results for next `before_model` |

This is cleaner than having separate middleware for each concern!

In [ ]:
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.tools.tool_node import ToolCallRequest
from langchain_core.messages import ToolMessage
from langgraph.runtime import Runtime
from typing import Callable, Union, Any

class DocumentGradingMiddleware(AgentMiddleware[CorrectiveRAGState]):
    """Middleware that grades RAG tool outputs for relevance.
    
    Used by: BOTH Wrap-Style and Node-Style agents.
    
    Pattern:
    - wrap_tool_call: Execute tool, grade documents, store results in instance
    - before_model: Apply stored grading results to state
    
    WHY this pattern? wrap_tool_call MUST return ToolMessage, not state updates.
    We store grading results temporarily, then apply them in before_model which
    CAN return state update dicts.
    """
    
    state_schema = CorrectiveRAGState
    
    def __init__(self, grader, rag_tool_name: str = "search_nike_report"):
        self.grader = grader
        self.rag_tool_name = rag_tool_name
        # WHY instance variable? wrap_tool_call can't return state updates,
        # so we store results here and apply them in the next before_model call.
        self._pending_grading: dict | None = None
    
    def before_model(self, state: CorrectiveRAGState, runtime: Runtime) -> dict[str, Any] | None:
        """Apply pending grading results to state and log."""
        num_messages = len(state.get("messages", []))
        
        print(f"\n{'='*60}")
        print(f"🤖 Model Call (messages: {num_messages})")
        print(f"   State:")
        print(f"   - documents_relevant: {state.get('documents_relevant')}")
        print(f"   - used_web_fallback: {state.get('used_web_fallback', False)}")
        
        # Apply any pending grading results from wrap_tool_call
        if self._pending_grading is not None:
            updates = self._pending_grading
            self._pending_grading = None  # Clear after applying
            print(f"   📊 Applying grading update: documents_relevant={updates.get('documents_relevant')}")
            return updates
        
        return None
    
    def after_model(self, state: CorrectiveRAGState, runtime: Runtime) -> dict[str, Any] | None:
        """Log after model invocation."""
        last_msg = state.get("messages", [])[-1] if state.get("messages") else None
        
        if last_msg:
            has_tool_calls = hasattr(last_msg, "tool_calls") and last_msg.tool_calls
            print(f"\n📤 Model Response:")
            print(f"   Type: {type(last_msg).__name__}")
            print(f"   Has tool calls: {has_tool_calls}")
            if has_tool_calls:
                for tc in last_msg.tool_calls:
                    print(f"   - Tool: {tc['name']}")
        print(f"{'='*60}")
        return None
    
    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage],
    ) -> ToolMessage:
        """Intercept tool calls to grade RAG outputs.
        
        Returns ToolMessage (NOT Command). State updates stored for before_model.
        """
        tool_name = request.tool_call.get("name", "")
        tool_args = request.tool_call.get("args", {})
        
        # Execute the tool first
        result = handler(request)
        
        # Only grade if this is our RAG tool
        if tool_name == self.rag_tool_name and isinstance(result, ToolMessage):
            query = tool_args.get("query", "")
            doc_content = result.content
            
            # Grade the documents
            try:
                grade_result = self.grader.invoke({
                    "question": query,
                    "document": doc_content[:2000]
                })
                
                # Handle potential None response from structured output
                if grade_result is None:
                    print(f"⚠️ Grading returned None - model may not support structured output well")
                    return result
                
                if not hasattr(grade_result, 'binary_score'):
                    print(f"⚠️ Grading result missing binary_score: {grade_result}")
                    return result
                
                is_relevant = grade_result.binary_score.lower() == "yes"
                reasoning = getattr(grade_result, 'reasoning', 'No reasoning provided')
                
                current_history = request.state.get("grading_history", [])
                new_history = current_history + [{
                    "query": query,
                    "relevant": is_relevant,
                    "reasoning": reasoning,
                }]
                
                print(f"\n📊 Document Grading:")
                print(f"   Query: {query}")
                print(f"   Relevant: {is_relevant}")
                print(f"   Reasoning: {reasoning}")
                
                # Store grading results - will be applied in next before_model
                self._pending_grading = {
                    "last_rag_query": query,
                    "documents_relevant": is_relevant,
                    "grade_reasoning": reasoning,
                    "grading_history": new_history,
                }
                
            except Exception as e:
                print(f"⚠️ Grading failed: {e}")
        
        return result

print("✅ DocumentGradingMiddleware defined (used by BOTH styles)")

✅ DocumentGradingMiddleware defined (used by BOTH styles)


### ⚠️ Grading Reliability Note

Structured output grading can fail with some models. The middleware handles this by checking for `None` and missing attributes before accessing `binary_score`. If grading fails, the agent continues without updating state - the model may retry with a different query.

## 🔀 Step 6: Web Search Fallback Middleware

This middleware performs web search fallback when RAG documents are irrelevant:

| Hook | Purpose |
|------|---------|
| `before_model` | Check state, call Tavily if needed, inject results as message |

**Key Pattern**: Manually call the fallback tool and inject results via `before_model`, since `request.override(tools=...)` doesn't actually filter tool access.

In [11]:
from langchain_core.messages import HumanMessage, AIMessage

class WebSearchFallbackMiddleware(AgentMiddleware):
    """Middleware that performs web search fallback when RAG documents are irrelevant.
    
    Used by: Node-Style agent ONLY.
    
    Manually calls web search and injects results via before_model.
    This gives deterministic fallback behavior - middleware decides, not the model.
    """
    
    state_schema = CorrectiveRAGState
    
    def __init__(self, web_search_tool):
        self.web_search_tool = web_search_tool
    
    def _get_original_user_query(self, state: CorrectiveRAGState) -> str | None:
        """Extract the original user question from messages."""
        messages = state.get("messages", [])
        for msg in messages:
            # Find the first HumanMessage (original user query)
            if hasattr(msg, "type") and msg.type == "human":
                return msg.content
            # Also handle dict format
            if isinstance(msg, dict) and msg.get("role") == "user":
                return msg.get("content")
        return None
    
    def before_model(self, state: CorrectiveRAGState, runtime: Runtime) -> dict[str, Any] | None:
        """When RAG failed, perform web search and inject results."""
        
        documents_relevant = state.get("documents_relevant")
        used_web_fallback = state.get("used_web_fallback", False)
        last_rag_query = state.get("last_rag_query")
        
        should_search = (
            documents_relevant is False and 
            not used_web_fallback and
            last_rag_query
        )
        
        if should_search:
            # Use original user query for better search results
            original_query = self._get_original_user_query(state)
            search_query = original_query or last_rag_query
            
            print(f"\n🌐 RAG failed - performing web search")
            print(f"   Original query: {original_query}")
            print(f"   Search query: {search_query}")
            
            try:
                web_result = self.web_search_tool.invoke(search_query)
                web_result_str = str(web_result)[:1500]
                
                print(f"   ✅ Web search completed")
                print(f"   📄 Preview: {web_result_str[:200]}...")
                
                injection_msg = AIMessage(content=f"I found relevant web search results for your query:\n\n{web_result_str}\n\nLet me summarize this information for you.")
                
                return {
                    "used_web_fallback": True,
                    "messages": [injection_msg],
                }
                
            except Exception as e:
                print(f"   ❌ Web search failed: {e}")
                return {"used_web_fallback": True}
        
        return None

print("✅ WebSearchFallbackMiddleware defined (Node-Style only)")
print("   Now uses original user query for better web search results")

✅ WebSearchFallbackMiddleware defined (Node-Style only)
   Now uses original user query for better web search results


In [12]:
# Instantiate middleware (used by both Approach A and B agents)
grading_middleware = DocumentGradingMiddleware(
    grader=grader,
    rag_tool_name="search_nike_report"
)

fallback_middleware = WebSearchFallbackMiddleware(
    web_search_tool=tavily_search
)

# Base system prompt - generic, doesn't mention specific tool names
# This prevents model from "hallucinating" tool calls for filtered tools
SYSTEM_PROMPT = """You are a helpful assistant that answers questions about Nike's business.

Use the available tools to find information. Start with the Nike report search tool.
If that doesn't provide relevant results, additional tools may become available.

Base your answers on retrieved information. Keep answers concise (2-3 sentences) and cite sources."""

print("✅ Middleware instances created")
print("✅ System prompt defined")

✅ Middleware instances created
✅ System prompt defined


## 📚 Quick Reference: All Middleware Hooks

| Hook | When | Returns |
|------|------|---------|
| `before_agent` / `after_agent` | Start/end of agent run | `dict` for state updates |
| `before_model` / `after_model` | Before/after each LLM call | `dict` for state updates |
| `wrap_model_call` | Around LLM call | Must call `handler(request)` |
| `wrap_tool_call` | Around tool call | Must return `ToolMessage` |

**Best Practice**: Combine related hooks in one middleware class rather than separate classes.

## 🔬 Two Middleware Styles Comparison

| Style | Mechanism | Who Decides Fallback |
|-------|-----------|---------------------|
| **Wrap-Style** | `wrap_model_call` + `request.override(tools=...)` | Model chooses from filtered tools |
| **Node-Style** | Call tool directly in `before_model` | Middleware triggers automatically |

### Wrap-Style: Tool Filtering

```python
def wrap_model_call(self, request: ModelRequest, handler: Callable) -> ModelCallResult:
    selected_tools = [t for t in request.tools if t.name in selected_names]
    modified_request = request.override(tools=selected_tools)
    return handler(modified_request)  # CRITICAL: pass modified_request!
```

**Constraint**: Can only filter from tools registered with `create_agent(tools=[...])`.

### Node-Style: Manual Fallback

```python
def before_model(self, state, runtime) -> dict | None:
    if should_use_fallback(state):
        result = self.fallback_tool.invoke(query)  # Direct call
        return {"messages": [AIMessage(content=f"Fallback: {result}")], "used_fallback": True}
    return None
```

**Benefits**: Tool never in model's list, full control, deterministic behavior.

### When to Use Which

| Scenario | Style | Why |
|----------|-------|-----|
| Progressive disclosure | Wrap-Style | Start with basic tools, unlock advanced ones |
| Fallback/recovery | **Node-Style** | Fallback shouldn't be model's choice |
| Cost optimization | Wrap-Style | Prevent expensive tool calls |

### ❌ Common Wrap-Style Pitfalls

1. `request.override(tools=[unregistered_tool])` → `ValueError`
2. `return handler(request)` instead of `return handler(modified_request)`
3. Mentioning filtered tools in prompt → model "hallucinates" calls

### 📊 Node-Style Execution Flow

```mermaid
graph TD;
    A[User Query + Initial State] --> B[DocumentGradingMiddleware.before_model];
    B -->|Apply pending grading results| C[WebSearchFallbackMiddleware.before_model];
    C -->|If irrelevant: call Tavily, inject results| D[Model Invocation];
    D -->|Tool call?| E{Has Tool Call?};
    E -->|Yes| F;
    
    subgraph F[DocumentGradingMiddleware.wrap_tool_call]
        F1[Execute RAG Tool] --> F2[Grade Documents]
    end
    
    F2 -->|Store grading results| B;
    E -->|No| G[Final Response];
```

**Key difference**: `before_model` **automatically** triggers web search when documents are irrelevant - no model decision needed. The `wrap_tool_call` wraps tool execution to grade results.

In [13]:
# Approach A: Tool Filtering Middleware (wrap_model_call pattern)

from langchain.agents.middleware.types import ModelRequest, ModelResponse, ModelCallResult

class ToolFilteringMiddleware(AgentMiddleware):
    """Middleware that filters available tools based on state.
    
    Used by: Wrap-Style agent ONLY.
    
    Use Case: Register multiple tools but conditionally hide some from the model.
    Example: Start with only RAG tool, enable web search after RAG fails.
    
    Constraint: Can only filter from tools registered with create_agent(tools=[...]).
    """
    
    state_schema = CorrectiveRAGState
    
    def __init__(self, rag_tool_name: str, web_tool_name: str):
        self.rag_tool_name = rag_tool_name
        self.web_tool_name = web_tool_name
        self._web_search_called: bool = False
    
    def before_model(self, state: CorrectiveRAGState, runtime: Runtime) -> dict[str, Any] | None:
        """Apply web search tracking to state."""
        if self._web_search_called:
            self._web_search_called = False
            return {"used_web_fallback": True}
        return None
    
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelCallResult:
        """Filter tools based on current state."""
        
        documents_relevant = request.state.get("documents_relevant")
        used_web_fallback = request.state.get("used_web_fallback", False)
        
        should_enable_web = (documents_relevant is False and not used_web_fallback)
        
        if should_enable_web:
            filtered_tools = request.tools
            print(f"   🔓 Enabling all tools: {[t.name for t in filtered_tools]}")
        else:
            filtered_tools = [t for t in request.tools if t.name == self.rag_tool_name]
            print(f"   🔒 Filtering to: {[t.name for t in filtered_tools]}")
        
        modified_request = request.override(tools=filtered_tools)
        # CRITICAL: Pass modified_request to handler, not original request!
        return handler(modified_request)
    
    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage],
    ) -> ToolMessage:
        """Track when web search tool is called."""
        tool_name = request.tool_call.get("name", "")
        result = handler(request)
        
        if tool_name == self.web_tool_name:
            self._web_search_called = True
            print(f"   📝 Tracking: web search tool was called")
        
        return result

print("✅ ToolFilteringMiddleware defined (Wrap-Style only)")

✅ ToolFilteringMiddleware defined (Wrap-Style only)


### 📊 Wrap-Style Execution Flow

```mermaid
graph TD;
    A[User Query + Initial State] --> B[DocumentGradingMiddleware.before_model];
    B -->|Apply pending grading results| C[ToolFilteringMiddleware.before_model];
    C -->|Track web search usage| D;
    
    subgraph D[ToolFilteringMiddleware.wrap_model_call]
        D1[Filter tools based on state 🔒/🔓] --> D2[Model Invocation]
    end
    
    D2 -->|Tool call?| F{Has Tool Call?};
    F -->|Yes| G;
    
    subgraph G[DocumentGradingMiddleware.wrap_tool_call]
        G1[Execute Tool] --> G2[Grade if RAG / Track if Web]
    end
    
    G2 --> B;
    F -->|No| H[Final Response];
```

**Key difference**: Both `wrap_model_call` and `wrap_tool_call` are **wrappers** - they intercept before and after their target invocation. Model decides whether to use web search when tools are unlocked (🔓).

## 🚀 Step 7: Build Corrective RAG Agents

Create both agents for comparison:
- **Wrap-Style**: Both tools registered, middleware filters dynamically
- **Node-Style**: Only RAG tool registered, middleware calls fallback directly

In [14]:
from langchain.agents import create_agent

# Tools for each style
rag_tools = [search_nike_report]  # Node-style: only RAG
all_tools = [search_nike_report, tavily_search]  # Wrap-style: both tools

# Create filtering middleware for Wrap-style
tool_filtering_middleware = ToolFilteringMiddleware(
    rag_tool_name="search_nike_report",
    web_tool_name="tavily_search_results_json"
)

# ==================== Wrap-Style (Tool Filtering) ====================
approach_a_agent = create_agent(
    model=chat_model,
    tools=all_tools,  # Both tools registered
    system_prompt=SYSTEM_PROMPT,
    middleware=[
        grading_middleware,          # Grades RAG results
        tool_filtering_middleware,   # Filters tools based on state
    ],
    state_schema=CorrectiveRAGState,
)

# ==================== Node-Style (Manual Fallback) ====================
approach_b_agent = create_agent(
    model=chat_model,
    tools=rag_tools,  # Only RAG tool registered
    system_prompt=SYSTEM_PROMPT,
    middleware=[
        grading_middleware,      # Grades RAG results
        fallback_middleware,     # Manually calls web search when RAG fails
    ],
    state_schema=CorrectiveRAGState,
)

print("✅ Both agents created")
print("\n📦 Wrap-Style (Tool Filtering):")
print(f"   Tools: {[t.name for t in all_tools]}")
print("   Middleware: grading + tool_filtering (wrap_model_call)")
print("\n📦 Node-Style (Manual Fallback):")
print(f"   Tools: {[t.name for t in rag_tools]}")
print("   Middleware: grading + fallback (before_model)")

✅ Both agents created

📦 Wrap-Style (Tool Filtering):
   Tools: ['search_nike_report', 'tavily_search_results_json']
   Middleware: grading + tool_filtering (wrap_model_call)

📦 Node-Style (Manual Fallback):
   Tools: ['search_nike_report']
   Middleware: grading + fallback (before_model)


## 🧪 Test the Corrective RAG System

### 👀 What to Observe in Test Output

When running tests, look for these key differences:

| Signal | Wrap-Style | Node-Style |
|--------|------------|------------|
| **Tool unlocking** | `🔓 Enabling all tools` when docs irrelevant | Never shows (web search not registered) |
| **Fallback trigger** | Model chooses `tavily_search_results_json` | `🌐 RAG failed - performing web search` automatic |
| **Determinism** | Model may skip web search even when available | Always triggers on irrelevant docs |

**Tip**: Compare the same question on both agents - Node-Style always falls back deterministically.

### Test 1: Nike Question with Relevant Documents

Test both approaches with a question that should find relevant documents in the RAG system.

In [15]:
# Helper function to run test on both agents
def run_comparison_test(question: str, test_name: str):
    """Run the same question on both agents and compare results."""
    
    initial_state = {
        "documents_relevant": None,
        "used_web_fallback": False,
        "grading_history": [],
    }
    
    print(f"{'='*80}")
    print(f"📝 {test_name}")
    print(f"❓ Question: {question}")
    print(f"{'='*80}\n")
    
    # Reset middleware state
    grading_middleware._pending_grading = None
    tool_filtering_middleware._web_search_called = False
    
    # ==================== Wrap-Style (Tool Filtering) ====================
    print("🅰️ WRAP-STYLE (Tool Filtering)")
    print("-" * 40)
    result_wrap = approach_a_agent.invoke({
        "messages": [{"role": "user", "content": question}],
        **initial_state
    })
    print(f"\n💬 Answer: {result_wrap['messages'][-1].content[:500]}...")
    print(f"📊 State: relevant={result_wrap.get('documents_relevant')}, fallback={result_wrap.get('used_web_fallback')}")
    
    # Reset middleware state
    grading_middleware._pending_grading = None
    
    # ==================== Node-Style (Manual Fallback) ====================
    print(f"\n{'='*40}")
    print("🅱️ NODE-STYLE (Manual Fallback)")
    print("-" * 40)
    result_node = approach_b_agent.invoke({
        "messages": [{"role": "user", "content": question}],
        **initial_state
    })
    print(f"\n💬 Answer: {result_node['messages'][-1].content[:500]}...")
    print(f"📊 State: relevant={result_node.get('documents_relevant')}, fallback={result_node.get('used_web_fallback')}")
    
    return result_wrap, result_node

# Run Test 1: Relevant documents
print("🧪 Test 1: Nike business question (should find relevant docs)\n")
result_wrap_1, result_node_1 = run_comparison_test(
    "What are Nike's main revenue sources and business segments?",
    "Test 1: Relevant Documents"
)

🧪 Test 1: Nike business question (should find relevant docs)

📝 Test 1: Relevant Documents
❓ Question: What are Nike's main revenue sources and business segments?

🅰️ WRAP-STYLE (Tool Filtering)
----------------------------------------

🤖 Model Call (messages: 1)
   State:
   - documents_relevant: None
   - used_web_fallback: False
   🔒 Filtering to: ['search_nike_report']

📤 Model Response:
   Type: AIMessage
   Has tool calls: [{'name': 'search_nike_report', 'args': {'query': 'revenue sources business segments operations financial performance'}, 'id': '019ae40cd9eb42f09dd4d4e29e3e5c03', 'type': 'tool_call'}]
   - Tool: search_nike_report

📊 Document Grading:
   Query: revenue sources business segments operations financial performance
   Relevant: True
   Reasoning: Document 1 is highly relevant as it directly addresses revenue sources, business segments, and operating segments with detailed financial breakdown by geographic regions (North America, Europe/Middle East/Africa, Greater C

### Test 2: Question That Triggers Web Fallback

Test both approaches with a question about recent events (not in 10-K report) - should trigger web search fallback.

In [16]:
# Run Test 2: Web fallback
# Query is crafted to sound like it might be in 10-K, but requires recent data
print("🧪 Test 2: Question requiring recent data (should trigger web fallback)\n")
result_wrap_2, result_node_2 = run_comparison_test(
    "What were Nike's quarterly earnings and revenue growth in Q3 2024?",
    "Test 2: Web Search Fallback"
)

🧪 Test 2: Question requiring recent data (should trigger web fallback)

📝 Test 2: Web Search Fallback
❓ Question: What were Nike's quarterly earnings and revenue growth in Q3 2024?

🅰️ WRAP-STYLE (Tool Filtering)
----------------------------------------

🤖 Model Call (messages: 1)
   State:
   - documents_relevant: None
   - used_web_fallback: False
   🔒 Filtering to: ['search_nike_report']

📤 Model Response:
   Type: AIMessage
   Has tool calls: [{'name': 'search_nike_report', 'args': {'query': 'Q3 2024 quarterly earnings revenue growth'}, 'id': '019ae40d3840c7585f65b5ef5ccaa179', 'type': 'tool_call'}]
   - Tool: search_nike_report

📊 Document Grading:
   Query: Q3 2024 quarterly earnings revenue growth
   Relevant: False
   Reasoning: The documents contain financial data for fiscal 2023, not Q3 2024 quarterly earnings or revenue growth. The data shown is annual revenue figures ($51.2 billion for fiscal 2023) and regional breakdowns, but there is no specific information about Q3 2024 

### 📊 Test Results: Wrap-Style vs Node-Style

| Aspect | Wrap-Style (Tool Filtering) | Node-Style (Manual Fallback) |
|--------|----------------------------|------------------------------|
| **Who decides web search?** | Model chooses from filtered tools | Middleware triggers automatically |
| **Search query** | Model crafts optimized query | Uses original user query |
| **Flexibility** | ✅ High - model adapts query | ⚠️ Lower - fixed query |
| **Determinism** | ⚠️ Non-deterministic (model may skip) | ✅ Deterministic (always triggers) |
| **Control** | Less control - model decides | Full control - middleware decides |

### 🔍 Key Observation from Test 2

| Style | Web Search Query | Result Quality |
|-------|------------------|----------------|
| **Wrap-Style** | `"Nike Q3 2024 quarterly earnings revenue growth results"` (model-crafted) | ✅ Detailed: $12.43B revenue, $0.98 EPS, 24% earnings growth |
| **Node-Style** | `"What were Nike's quarterly earnings..."` (raw user query) | ⚠️ Good but less specific |

### 💡 Recommendation

| Use Case | Recommended Style | Why |
|----------|-------------------|-----|
| **Flexible search needed** | **Wrap-Style** | Model crafts optimized search queries |
| **Guaranteed fallback** | Node-Style | Deterministic, no model decision |
| **Cost-sensitive** | Wrap-Style | Can prevent unnecessary tool calls |
| **Simple/predictable flow** | Node-Style | Easier to debug and test |

**Best Practice**: For production Corrective RAG, consider **Wrap-Style** for its flexibility - the model can craft better search queries based on context. Use Node-Style only when you need guaranteed deterministic fallback regardless of model behavior.

### 🔧 Potential Node-Style Improvement

To get the best of both worlds, Node-Style could be enhanced to have the model generate an optimized search query before fallback:

```python
# In WebSearchFallbackMiddleware.before_model:
# Instead of using raw user query, ask model to craft search query
optimized_query = llm.invoke(f"Generate a web search query for: {original_query}")
web_result = self.web_search_tool.invoke(optimized_query)
```

## 📝 Summary

### Key Takeaways

| Concept | Implementation |
|---------|----------------|
| **State Schema** | Use `AgentState` (TypedDict) for mutable state |
| **State Updates** | Return `dict` from `before_model`/`after_model` |
| **Tool Interception** | `wrap_tool_call` returns `ToolMessage`, stores data in instance |
| **Fallback Tools** | Call manually in `before_model`, inject results as messages |
| **Tool Filtering** | Use `wrap_model_call` with `request.override(tools=...)` |

### Two Middleware Styles

| Style | Best For | Trade-off |
|-------|----------|-----------|
| **Wrap-Style** | Production CRAG, flexible search | Model crafts queries ✅, but may skip fallback ⚠️ |
| **Node-Style** | Guaranteed fallback, simple flow | Deterministic ✅, but less flexible queries ⚠️ |

**Recommendation**: Use **Wrap-Style** for most Corrective RAG implementations - the model's ability to craft optimized search queries typically outweighs the risk of skipping fallback.

### Corrective RAG Flow

```mermaid
graph LR;
    A[Query] --> B[RAG Tool];
    B --> C[Grade];
    C -->|Relevant| D[Generate Answer];
    C -->|Irrelevant| E[Web Search];
    E --> D;
```

## 🔍 LangSmith Tracing

### How to View Traces:

1. **Enable LangSmith** in your `.env` file:
   ```bash
   LANGCHAIN_TRACING_V2=true
   LANGCHAIN_API_KEY=your-api-key
   LANGCHAIN_PROJECT=ep2-langchain-retrieval
   ```

2. **Re-run the tests above** - Traces will be automatically captured

3. **Visit LangSmith** at https://smith.langchain.com

### What to Look For:

**In Middleware CRAG traces, you'll see:**
- 🤖 Model call with before/after hooks
- 📚 RAG tool execution
- ✅ Grading invocation (nested under tool call)
- 🌐 Web search tool (if documents were irrelevant)
- ✨ Final response generation